In [1]:
#Cell 1
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

# Set deterministic seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# File paths
input_path = "dataset/original/EVSE-B-PowerCombined_filtered.csv"
output_dir = "dataset/synthetic/"
os.makedirs(output_dir, exist_ok=True)

# Select required features and target
selected_features = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW", "State"]
target_col = "Attack"

# Load dataset
df = pd.read_csv(input_path)

# Filter required columns
columns_to_keep = selected_features + [target_col]
df_filtered = df[columns_to_keep].copy()

print(f"Data loaded successfully. Shape: {df_filtered.shape}")
print(df_filtered.head())

Data loaded successfully. Shape: (49017, 6)
   shunt_voltage  bus_voltage_V  current_mA  power_mW State     Attack
0            978          5.165        1027      5300  idle  syn-flood
1            872          5.161        1009      4980  idle  syn-flood
2           1017          5.165        1029      5300  idle  syn-flood
3            930          5.161        1005      5180  idle  syn-flood
4            958          5.165        1034      5180  idle  syn-flood


In [2]:
#Cell 2
# Convert categorical column 'State' (idle, charging) via One-Hot Encoding
df_encoded = pd.get_dummies(df_filtered, columns=["State"], drop_first=False)

# Store feature names after encoding (input_dim = 6)
feature_cols = [c for c in df_encoded.columns if c != target_col]
input_dim = len(feature_cols)

print(f"Input Features ({input_dim}): {feature_cols}")

# Fit class-wise Scalers to preserve distinct statistical distributions per attack group
scalers = {}
processed_data_by_class = {}

for attack_class in df_encoded[target_col].unique():
    class_df = df_encoded[df_encoded[target_col] == attack_class].copy()
    
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(class_df[feature_cols])
    
    scalers[attack_class] = scaler
    processed_data_by_class[attack_class] = scaled_features

print("Preprocessing complete. Scalers fitted per Attack class.")

Input Features (6): ['shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW', 'State_charging', 'State_idle']
Preprocessing complete. Scalers fitted per Attack class.


In [3]:
# Cell 3
class TabularDAE(nn.Module):
    """
    Denoising Autoencoder updated with specific hyperparameters:
    - alpha: 0.75, beta: 1
    - input_dim: 6, hidden_dims: (32, 16), latent_dim: 18
    - dropout: 0.05
    - batch_size: 64, learning_rate: 0.002, weight_decay: 0
    - noise_factor: 0.03, denoise_val: True, denoise_test: False
    """
    
    def __init__(
        self,
        input_dim=6,
        hidden_dims=(32, 16),
        latent_dim=18,
        dropout=0.05,
        alpha=0.75,
        beta=1.0,
        batch_size=64,
        learning_rate=0.002,
        weight_decay=0.0,
        noise_factor=0.03,
        denoise_val=True,
        denoise_test=False
    ):
        super(TabularDAE, self).__init__()
        
        """
    def __init__(
        self,
        input_dim=6,
        hidden_dims=(32, 16),
        latent_dim=16,
        dropout=0.05,
        alpha=1.0,
        beta=1.0,
        batch_size=64,
        learning_rate=0.002,
        weight_decay=0.0,
        noise_factor=0.3,
        denoise_val=True,
        denoise_test=False
    ):
        super(TabularDAE, self).__init__()
"""
        # Save training & evaluation hyperparameters as attributes
        self.input_dim = input_dim
        self.hidden_dims = hidden_dims
        self.latent_dim = latent_dim
        self.dropout_rate = dropout
        self.alpha = alpha
        self.beta = beta
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.noise_factor = noise_factor
        self.denoise_val = denoise_val
        self.denoise_test = denoise_test

        # Encoder: 6 -> 32 -> 16 -> 18
        self.encoder = nn.Sequential(
            nn.Linear(self.input_dim, self.hidden_dims[0]),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),
            nn.Linear(self.hidden_dims[0], self.hidden_dims[1]),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),
            nn.Linear(self.hidden_dims[1], self.latent_dim)
        )

        # Decoder: 18 -> 16 -> 32 -> 6
        self.decoder = nn.Sequential(
            nn.Linear(self.latent_dim, self.hidden_dims[1]),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),
            nn.Linear(self.hidden_dims[1], self.hidden_dims[0]),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),
            nn.Linear(self.hidden_dims[0], self.input_dim)
        )

    def add_noise(self, x):
        """Applies Gaussian noise to input tensor based on noise_factor."""
        if self.noise_factor > 0:
            noise = torch.randn_like(x) * self.noise_factor
            return x + noise
        return x

    def forward(self, x, apply_noise=False):
        """Forward pass with optional noise injection."""
        if apply_noise:
            x = self.add_noise(x)
        
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        return reconstruction

print("DAE Architecture defined.")

DAE Architecture defined.


In [4]:
# Cell 4 (modified with model saving)
# Training settings
best_epoch = 200  # Set desired maximum epochs
models_by_class = {}
saved_models = {}

# Train DAE for each attack class using preprocessed data from Cell 2
for attack_class, data in processed_data_by_class.items():
    print(f"\n--- Training DAE for Attack Class: '{attack_class}' ---")
    
    tensor_data = torch.tensor(data, dtype=torch.float32)
    dataset = TensorDataset(tensor_data)
    
    # Initialize DAE model (uses feature_cols count and hyperparameters defined in Cell 3)
    model = TabularDAE(input_dim=len(feature_cols)).to(device)
    
    # Create DataLoader
    dataloader = DataLoader(dataset, batch_size=model.batch_size, shuffle=True, drop_last=False)
    
    # Optimizer using model hyperparameters
    optimizer = optim.Adam(
        model.parameters(), 
        lr=model.learning_rate, 
        weight_decay=model.weight_decay
    )
    criterion = nn.MSELoss()
    
    model.train()
    for epoch in range(1, best_epoch + 1):
        total_loss = 0.0
        for batch in dataloader:
            x_clean = batch[0].to(device)
            
            # Forward pass with noise injection
            optimizer.zero_grad()
            reconstructed = model(x_clean, apply_noise=True)
            
            loss = criterion(reconstructed, x_clean)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item() * x_clean.size(0)
            
        epoch_loss = total_loss / len(dataset)
        if epoch % 10 == 0 or epoch == best_epoch:
            print(f"Class: {attack_class} | Epoch {epoch}/{best_epoch} | Loss: {epoch_loss:.6f}")
    
    # Store trained model
    models_by_class[attack_class] = model
    
    # === Save the trained DAE model ===
    model_path = f"models/dae_{attack_class}.pt"
    os.makedirs("models", exist_ok=True)
    
    # Save complete model state (architecture + weights + hyperparameters)
    torch.save({
        'epoch': best_epoch,
        'model_state_dict': model.state_dict(),
        'model_config': {
            'input_dim': model.input_dim,
            'hidden_dims': model.hidden_dims,
            'latent_dim': model.latent_dim,
            'dropout': model.dropout_rate,
            'alpha': model.alpha,
            'beta': model.beta,
            'batch_size': model.batch_size,
            'learning_rate': model.learning_rate,
            'weight_decay': model.weight_decay,
            'noise_factor': model.noise_factor,
            'denoise_val': model.denoise_val,
            'denoise_test': model.denoise_test
        }
    }, model_path)
    
    saved_models[attack_class] = model_path
    print(f"✓ Saved DAE model for '{attack_class}' to: {model_path}")

print("\nAll DAE models trained successfully.")
print(f"\nSaved models: {saved_models}")


--- Training DAE for Attack Class: 'syn-flood' ---
Class: syn-flood | Epoch 10/200 | Loss: 0.029442
Class: syn-flood | Epoch 20/200 | Loss: 0.020992
Class: syn-flood | Epoch 30/200 | Loss: 0.019100
Class: syn-flood | Epoch 40/200 | Loss: 0.018426
Class: syn-flood | Epoch 50/200 | Loss: 0.018379
Class: syn-flood | Epoch 60/200 | Loss: 0.017371
Class: syn-flood | Epoch 70/200 | Loss: 0.016510
Class: syn-flood | Epoch 80/200 | Loss: 0.016567
Class: syn-flood | Epoch 90/200 | Loss: 0.015351
Class: syn-flood | Epoch 100/200 | Loss: 0.014598
Class: syn-flood | Epoch 110/200 | Loss: 0.013779
Class: syn-flood | Epoch 120/200 | Loss: 0.013327
Class: syn-flood | Epoch 130/200 | Loss: 0.013386
Class: syn-flood | Epoch 140/200 | Loss: 0.013622
Class: syn-flood | Epoch 150/200 | Loss: 0.013027
Class: syn-flood | Epoch 160/200 | Loss: 0.013168
Class: syn-flood | Epoch 170/200 | Loss: 0.013702
Class: syn-flood | Epoch 180/200 | Loss: 0.012595
Class: syn-flood | Epoch 190/200 | Loss: 0.012846
Class: 

In [5]:
#Cell 5
target_counts = {
    'none': 10053, 
    'Backdoor': 14796, 
    'syn-flood': 9463
}

synthetic_dfs = []

for attack_class, count in target_counts.items():
    if attack_class not in models_by_class:
        print(f"Warning: Class '{attack_class}' not found in trained models. Skipping.")
        continue
        
    model = models_by_class[attack_class]
    model.eval()
    
    # Draw original base samples and inject noise for generation
    original_data = processed_data_by_class[attack_class]
    indices = np.random.choice(len(original_data), size=count, replace=True)
    base_samples = torch.tensor(original_data[indices], dtype=torch.float32).to(device)
    
    with torch.no_grad():
        noisy_samples = base_samples + torch.randn_like(base_samples) * noise_factor
        synthetic_scaled = model(noisy_samples).cpu().numpy()
    
    # Inverse transform to original scale
    scaler = scalers[attack_class]
    synthetic_unscaled = scaler.inverse_transform(synthetic_scaled)
    
    syn_df = pd.DataFrame(synthetic_unscaled, columns=feature_cols)
    
    # Reverse One-Hot Encoding for 'State'
    state_cols = [c for c in feature_cols if c.startswith("State_")]
    if state_cols:
        syn_df["State"] = syn_df[state_cols].idxmax(axis=1).apply(lambda x: x.replace("State_", ""))
        syn_df.drop(columns=state_cols, inplace=True)
    
    syn_df[target_col] = attack_class
    synthetic_dfs.append(syn_df)

# Combine generated synthetic dataset
final_synthetic_df = pd.concat(synthetic_dfs, ignore_index=True)

# Save to output path
output_file = os.path.join(output_dir, "dae_synthetic_dataset.csv")
final_synthetic_df.to_csv(output_file, index=False)

print(f"Synthetic dataset saved to: {output_file}")
print("Generated target counts:")
print(final_synthetic_df[target_col].value_counts())
print("\nFirst 5 rows of generated data:")
print(final_synthetic_df.head())

NameError: name 'noise_factor' is not defined

In [ ]:
# Cell 6: Calculate Reconstruction Error and Plot ROC AUC Curves

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# 1. Define Normal vs. Attack ground truth labels
# Assuming 'none' represents normal operation and other classes are attacks
normal_class = "none"
classes = list(models_by_class.keys())

# Binarize labels for multi-class ROC AUC evaluation (One-vs-Rest)
y_true_labels = df_encoded[target_col].values
y_true_bin = label_binarize(y_true_labels, classes=classes)

# 2. Compute Reconstruction Loss using the Normal ('none') DAE Model
# Lower error = Normal, Higher error = Anomaly/Attack
normal_model = models_by_class[normal_class]
normal_scaler = scalers[normal_class]

normal_model.eval()

# Transform entire dataset using the normal class scaler
all_features_scaled = normal_scaler.transform(df_encoded[feature_cols])
x_tensor = torch.tensor(all_features_scaled, dtype=torch.float32).to(device)

with torch.no_grad():
    reconstructed_x = normal_model(x_tensor, apply_noise=False)
    # Compute per-sample Mean Squared Error (MSE) as anomaly score
    reconstruction_errors = torch.mean((x_tensor - reconstructed_x) ** 2, dim=1).cpu().numpy()

# 3. Calculate ROC and AUC for each class (One-vs-Rest based on Anomaly Scores)
plt.figure(figsize=(9, 6))

for i, class_name in enumerate(classes):
    # For attacks, higher loss = higher attack probability
    # For 'none' (normal), lower loss = higher normal probability
    if class_name == normal_class:
        scores = -reconstruction_errors
    else:
        scores = reconstruction_errors
        
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], scores)
    roc_auc = auc(fpr, tpr)
    
    plt.plot(fpr, tpr, lw=2, label=f"Class '{class_name}' (AUC = {roc_auc:.4f})")

# Plot baseline (random classifier)
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--", label="Random Chance (AUC = 0.50)")

# Formatting
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate (FPR)", fontsize=12)
plt.ylabel("True Positive Rate (TPR)", fontsize=12)
plt.title("ROC AUC Curve - DAE Reconstruction Anomaly Detection", fontsize=14)
plt.legend(loc="lower right", fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()

# Display the plot
plt.show()